# DriveValue — Notebook 1: Exploratory Data Analysis

**Project:** DriveValue — Nigerian Used Car Price Predictor  
**Dataset:** Nigerian Car Prices Dataset (Kaggle)  
**Target Variable:** `price`  

This notebook covers:
- Data loading and inspection
- Missing value analysis
- Distribution analysis
- Feature relationships
- Saving key visualizations for the app and README

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import warnings
warnings.filterwarnings('ignore')

sns.set(style='ticks', color_codes=True, font_scale=0.9)
plt.rcParams['figure.figsize'] = (10, 5)

os.makedirs('../assets', exist_ok=True)

print('Libraries loaded!')

## 1. Load Dataset

In [ ]:
path = kagglehub.dataset_download('makindekayode/nigerian-car-prices-dataset')
file_path = os.path.join(path, 'car_prices.csv')
df = pd.read_csv(file_path)

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 2. Basic Inspection

In [ ]:
print('Column names:')
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe().round(0)

## 3. Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(1)
})

print('=== MISSING VALUES ===')
print(missing_df[missing_df['Missing Count'] > 0])

## 4. Duplicate Check

In [ ]:
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')

## 5. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='price', bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Car Prices')
axes[0].set_xlabel('Price (N)')

sns.histplot(data=df, x=np.log1p(df['price']), bins=40, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Log Distribution of Car Prices')
axes[1].set_xlabel('Log Price')

plt.tight_layout()
plt.savefig('../assets/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean price:   N{df["price"].mean():>12,.0f}')
print(f'Median price: N{df["price"].median():>12,.0f}')
print(f'Min price:    N{df["price"].min():>12,.0f}')
print(f'Max price:    N{df["price"].max():>12,.0f}')

## 6. Mileage vs Price

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Mileage', y='price', alpha=0.4, color='steelblue')
plt.title('Mileage vs Price')
plt.xlabel('Mileage (km)')
plt.ylabel('Price (N)')
plt.tight_layout()
plt.savefig('../assets/mileage_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Year vs Average Price

In [ ]:
year_price = df.groupby('Year of manufacture')['price'].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(year_price['Year of manufacture'], year_price['price'], marker='o', color='steelblue')
plt.title('Average Car Price by Year of Manufacture')
plt.xlabel('Year')
plt.ylabel('Average Price (N)')
plt.tight_layout()
plt.savefig('../assets/year_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top Brands

In [ ]:
top_brands = df['Make'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='Blues_r')
plt.title('Top 10 Most Listed Car Brands')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 9. Price by Fuel Type and Gear Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='fuel type', y='price', showfliers=False, ax=axes[0])
axes[0].set_title('Price by Fuel Type')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Price (N)')

sns.boxplot(data=df, x='gear type', y='price', showfliers=False, ax=axes[1])
axes[1].set_title('Price by Gear Type')
axes[1].set_xlabel('Gear Type')
axes[1].set_ylabel('Price (N)')

plt.tight_layout()
plt.show()

## 10. Price by Condition

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Condition', y='price', showfliers=False)
plt.title('Price by Car Condition')
plt.xlabel('Condition')
plt.ylabel('Price (N)')
plt.tight_layout()
plt.show()

## 11. Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('../assets/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. EDA Summary

Key findings from the exploratory analysis:

1. **Price distribution** is heavily right-skewed — a small number of luxury/high-end cars have very high prices. Log transformation shows a more normal distribution.

2. **Mileage** shows a negative relationship with price — higher mileage cars tend to be cheaper, which makes sense as more driven cars depreciate more.

3. **Year of manufacture** shows a clear positive relationship with price — newer cars are worth more.

4. **Toyota, Honda and Mercedes** are the most listed brands in the dataset.

5. **Condition** significantly affects price — Brand New cars command the highest prices, followed by Foreign Used, then Nigerian Used.

6. **Fuel type** affects price — petrol is most common but diesel and hybrid cars tend to be pricier.

These findings directly inform the feature selection in Notebook 2.